# GetCompoundIdentifiers
This notebook is used to get a bunch of molecular identifiers <u>starting from SMILES strings</u>. These include:

1. [PubChem CIDs](https://pubchem.ncbi.nlm.nih.gov/docs/compounds)
2. InChI key
3. InChI
4. CAS number

## Imports

In [ ]:
# std lib
import sys
import urllib
import logging
from pathlib import Path

# Data wrangling
import pandas as pd

# Custom
from CommercialCompoundSearcher.file_io import read_table_file

# PubChem querying
from CommercialCompoundSearcher.query import query_cid_from_inchi_key
from CommercialCompoundSearcher.query import query_CAS_from_cid

# Database management
from CommercialCompoundSearcher.database import connect_pubchem_cache
from CommercialCompoundSearcher.database import close_pubchem_cache
from CommercialCompoundSearcher.database import get_CID_from_identifier
from CommercialCompoundSearcher.database import get_identifiers_from_CID
from CommercialCompoundSearcher.database import upsert_identifier_lookup
from CommercialCompoundSearcher.database import upsert_compound

# Utils
from CommercialCompoundSearcher.utils import remove_duplicate_values
from CommercialCompoundSearcher.utils import canonicalize_smiles, smiles_to_inchi_key, smiles_to_inchi

logger = logging.getLogger(__name__)

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='[%(levelname)-5s - %(asctime)s] %(message)s',
    datefmt='%m/%d/%Y:%H:%M:%S',  # Correct way to format the date
    handlers=[logging.StreamHandler(sys.stdout)]
)

## Reading in SMILES strings and default settings

The SMILES can be provided as a plaintext document (.txt) with "__SMILES__" in the first line and or a '.csv/.xlsx' spreadsheet with a "__SMILES__" column.

```
┌────────────────────────┐
│ SMILES                 │
│ CC(=O)OCC[N+](C)(C)C   │
│ CC(C[N+](C)(C)C)OC(=O) │
│ ...                    │
└────────────────────────┘

```


In [ ]:
# Define a file that contains your SMILES strings
file = Path('./data/smiles_small.txt')

# Define a directory to store the results
results_dir = Path('./results/')
results_dir.mkdir(exist_ok=True, parents=True)

# Define path to SQLite cache
# This stores the results to expedite restarts
db_path = Path('./data/pubchem_cache.sqlite')

# Control how often database commits (lower values are more frequent)
commit_every = 10

# Whether or not to drop empty rows in the initial DataFrame
drop_empty_rows = False

# Read in the file
df = read_table_file(file,
                     sheet_name=None)

if drop_empty_rows:
    df.dropna(axis=0, how='any', inplace=True)

display(df)

## Canonicalization and molecular identifiers

This section canonicalizes the SMILES strings and adds additional molecular identifiers using RDKit. The output of this block will contain warnings (and potentially errors) from RDKit. Many of these errors (such as None mol from RDKit) are handled by removing the SMILES string and storing it in a separate file. 

In [ ]:
# Apply canonicalization
#TODO Understand how this affects stereoisomerism in the SMILES/InChI/InChI key values
df['SMILES'] = df['SMILES'].apply(canonicalize_smiles)

# Add InChI column
df['INCHI'] = df['SMILES'].apply(smiles_to_inchi)

# Add InChI key column
df['INCHI_KEY'] = df['SMILES'].apply(smiles_to_inchi_key)

# Get every row that has np.nan values
failed = df[(df['INCHI'].isna()) | (df['INCHI_KEY'].isna())]

# Get every row that does not have np.nan values
df = df[~(df['INCHI'].isna()) | ~(df['INCHI_KEY'].isna())].copy(deep=True)

# Save the failed and the canonicalized datasets to a csv file
failed.to_csv(results_dir / 'failed_canonicalization.csv', index=False)
df.to_csv(results_dir / 'canonicalized.csv', index=False)

# Check if anything failed an notify the user
if not failed.empty:
    print(f'Some rows failed canonicalization. See {results_dir.resolve()} for details.')
    display(failed)

# Remove exact duplicates
df, duplicates = remove_duplicate_values(df=df, on='INCHI_KEY')

# For your viewing pleasure
display(df)

if duplicates.empty:
    print('No duplicate entries were found.')
else:
    display(duplicates)

# Save the results for good book keeping.
df.to_csv(results_dir / 'added_molecular_identifiers.csv', index=False)
duplicates.to_csv(results_dir / 'duplicate_molecular_identifiers.csv', index=False)

## Query Pubchem for CID

The best identifier to use for querying Pubchem is the [Pubchem Compound ID (CID)](https://pubchem.ncbi.nlm.nih.gov/docs/compounds). This section will obtain a CID for every InChI key, which we will use to get the CAS in a later section.

In [ ]:
# Get InChI keys as a list
inchi_keys = df['INCHI_KEY'].to_list()

# Get the total length of InChI keys for tracking progress
total = len(inchi_keys)

# Make a column for CID as integer type
df['CID'] = pd.Series(None, index=df.index, dtype='Int64')

# Open the cache database
conn = connect_pubchem_cache(db_path)

try:
    for i, inchi_key in enumerate(inchi_keys, start=1):

        logger.info('Working on %d of %d (%.2f%%)', i, total, i / total * 100)

        # Try cache first using the unified identifier lookup table
        cid, status = get_CID_from_identifier(conn=conn,
                                              identifier_type='inchikey',
                                              identifier_value=inchi_key)

        # Successful lookup of the CID
        if status == 'ok' and cid is not None:
            df.loc[df['INCHI_KEY'] == inchi_key, 'CID'] = int(cid)
            continue

        # Validate status. Everything past this block goes to PubChem query
        if status not in ['ok', 'not_found', 'http_error', 'parse_error', None]:
            raise ValueError(f'Unexpected cache status "{status}"')

        # Query PubChem
        try:
            cid = query_cid_from_inchi_key(inchi_key)
        except urllib.error.HTTPError as e:

            # Cache the failure
            upsert_identifier_lookup(conn=conn,
                                     identifier_type='inchikey',
                                     identifier_value=inchi_key,
                                     cid=None,
                                     status='http_error')

            logger.error('Could not get CID from InChI key %s because', inchi_key)
            logger.error(str(e))
            continue

        # Normalize / handle empty results
        if cid is None:
            upsert_identifier_lookup(conn=conn,
                                     identifier_type='inchikey',
                                     identifier_value=inchi_key,
                                     cid=None,
                                     status='not_found')

            # Store in the DataFrame as None so that it is an empty cell
            df.loc[df['INCHI_KEY'] == inchi_key, 'CID'] = None

        else:
            cid = int(cid)

            # Ensure parent compound row exists before storing the lookup
            upsert_compound(conn=conn,
                            cid=cid,
                            inchikey=inchi_key)

            # Cache the InChIKey to CID mapping
            upsert_identifier_lookup(conn=conn,
                                     identifier_type='inchikey',
                                     identifier_value=inchi_key,
                                     cid=cid,
                                     status='ok')

            df.loc[df['INCHI_KEY'] == inchi_key, 'CID'] = cid

        # Commit periodically
        if i % commit_every == 0:
            conn.commit()

    # Final commit
    conn.commit()

finally:
    close_pubchem_cache(conn)

# Show updated DataFrame
display(df)

## Query Pubchem for CAS number

Often a molecule will have multiple CAS numbers. The current implementation using `query_CAS_from_CID` will return only the first CAS number found. Users should be aware that this looks for two dashes (-) in the numbers it receives from Pubchem to identify the CAS number. This procedure could be improved by a more systematic way of determining whether the item received from Pubchem is actually a CAS number. The REST queries each take at least 200 ms.

__If you stop this cell while it is running, you will lose all of your progress towards acquiring vendors__

In [ ]:
# Get CIDs as a list
cids = df['CID'].astype('Int64').dropna().astype(int).to_list()

# Precompute CID series once
cid_series = df['CID'].astype('Int64')

# Get the total length of CIDs for tracking progress
total = len(cids)

# Open the cache database
conn = connect_pubchem_cache(db_path)

try:
    for i, cid in enumerate(cids, start=1):

        logger.info('Working on %d of %d (%.2f%%)', i, total, i / total * 100)

        cas = None

        # Try cache first by looking for cached CAS values mapped to this CID
        cas_numbers, status = get_identifiers_from_CID(conn=conn,
                                                       identifier_type='cas',
                                                       cid=cid)

        # Use cached CAS if available
        if status == 'ok' and cas_numbers is not None:
            if len(cas_numbers) > 1:
                logger.warning('Found %d CAS numbers for CID %d. %s',
                               len(cas_numbers),
                               cid,
                               str(cas_numbers))

            cas = cas_numbers[0]

        # Query PubChem if we do not have any cached CAS numbers
        else:
            try:
                cas = query_CAS_from_cid(cid)
            except urllib.error.HTTPError as e:
                logger.error('Could not get CAS from CID %d because', cid)
                logger.error(str(e))
                continue

            if cas is not None:
                # Ensure parent compound row exists before storing the lookup
                upsert_compound(conn=conn,
                                cid=cid)

                # Cache the CAS to CID mapping
                upsert_identifier_lookup(conn=conn,
                                         identifier_type='cas',
                                         identifier_value=str(cas),
                                         cid=cid,
                                         status='ok')

        # Store in the DataFrame
        if cas is not None:
            df.loc[cid_series == cid, 'CAS'] = str(cas)
        else:
            df.loc[cid_series == cid, 'CAS'] = None

        # Commit periodically
        if i % commit_every == 0:
            conn.commit()

    # Final commit
    conn.commit()

finally:
    close_pubchem_cache(conn)

# Show updated DataFrame
display(df)

## Save the results

In [ ]:
# Define the Path to the output file
destination = Path(results_dir / 'all_molecular_identifiers.csv')

# Reorder the columns how I like them
df.set_index('CID', inplace=True, drop=True)
df = df[['CAS', 'INCHI_KEY', 'SMILES', 'INCHI']]

df.to_csv(destination)